In [1]:
#volvemos a partir de tracks_base.parquet
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from functools import reduce
import math


In [2]:
spark = (
    SparkSession.builder
    .appName("HitMakerAI_HitScore_V2")
    .getOrCreate()
)

tracks_base = spark.read.parquet("../processed/tracks_base.parquet")

tracks_base.select(
    F.count("*").alias("total_filas"),
    F.countDistinct("id").alias("ids_unicos")
).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/20 23:26:08 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/20 23:26:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 23:26:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 1:>                                                          (0 + 8) / 8]

+-----------+----------+
|total_filas|ids_unicos|
+-----------+----------+
|     586264|    586264|
+-----------+----------+



# Hit Score V2: versión ponderada y ajustada temporalmente

En esta sección se construye una segunda versión del `Hit Score`.

La primera versión del score usaba un promedio simple de cinco componentes acústicos. Aunque esa aproximación era interpretable, tenía tres limitaciones principales:

1. Todas las variables tenían el mismo peso, aunque no todas mostraban la misma relación con `popularity`.
2. La normalización min-max podía ser sensible a valores extremos.
3. El score no consideraba el año de lanzamiento, por lo que canciones muy antiguas podían aparecer como recomendaciones principales.

Para mejorar la métrica, esta versión incorpora:

- diagnóstico de percentiles para detectar posibles valores extremos;
- normalización robusta usando clipping por percentiles;
- ponderación de variables según su correlación absoluta con `popularity`;
- una versión ajustada por recencia para evitar que las recomendaciones estén dominadas por canciones demasiado antiguas.

El objetivo no es predecir éxito comercial real, sino construir una métrica académica más robusta para ordenar canciones según su similitud acústica con canciones populares.

In [3]:
selected_features = [
    "danceability",
    "energy",
    "loudness",
    "acousticness",
    "instrumentalness"
]

In [4]:
quantile_probs = [0.01, 0.05, 0.50, 0.95, 0.99]

percentile_rows = []

for feature in selected_features:
    q = tracks_base.approxQuantile(feature, quantile_probs, 0.001)
    
    stats = tracks_base.select(
        F.min(feature).alias("min_value"),
        F.max(feature).alias("max_value")
    ).first()
    
    percentile_rows.append((
        feature,
        float(stats["min_value"]),
        float(q[0]),
        float(q[1]),
        float(q[2]),
        float(q[3]),
        float(q[4]),
        float(stats["max_value"])
    ))

percentile_diagnostics = spark.createDataFrame(
    percentile_rows,
    [
        "feature",
        "min_value",
        "p01",
        "p05",
        "p50",
        "p95",
        "p99",
        "max_value"
    ]
)

percentile_diagnostics.show(truncate=False)

+----------------+---------+-------+-------+-------+------+-----+---------+
|feature         |min_value|p01    |p05    |p50    |p95   |p99  |max_value|
+----------------+---------+-------+-------+-------+------+-----+---------+
|danceability    |0.0532   |0.161  |0.268  |0.577  |0.814 |0.891|0.991    |
|energy          |0.0      |0.0341 |0.12   |0.549  |0.93  |0.979|1.0      |
|loudness        |-55.0    |-26.018|-19.809|-9.245 |-3.923|-2.59|5.376    |
|acousticness    |0.0      |5.82E-5|0.00176|0.421  |0.983 |0.994|0.996    |
|instrumentalness|0.0      |0.0    |0.0    |2.42E-5|0.873 |0.942|1.0      |
+----------------+---------+-------+-------+-------+------+-----+---------+



El diagnóstico de percentiles muestra que algunas variables, especialmente `loudness`, presentan valores extremos en los mínimos y máximos. Por esta razón, en la versión 2 del Hit Score se aplica clipping entre los percentiles 1 y 99 antes de normalizar. Esto reduce la influencia de valores extremos y hace que la normalización sea más representativa para la mayoría de las canciones.

In [5]:
bounds = {
    row["feature"]: {
        "p01": row["p01"],
        "p99": row["p99"]
    }
    for row in percentile_diagnostics.collect()
}

tracks_scored_v2 = tracks_base

for feature in selected_features:
    lower = float(bounds[feature]["p01"])
    upper = float(bounds[feature]["p99"])

    tracks_scored_v2 = (
        tracks_scored_v2
        .withColumn(
            f"{feature}_clip",
            F.when(F.col(feature) < F.lit(lower), F.lit(lower))
             .when(F.col(feature) > F.lit(upper), F.lit(upper))
             .otherwise(F.col(feature))
        )
        .withColumn(
            f"{feature}_norm_v2",
            F.when(
                F.lit(upper) != F.lit(lower),
                (F.col(f"{feature}_clip") - F.lit(lower)) / F.lit(upper - lower)
            ).otherwise(F.lit(0.0))
        )
    )

A diferencia de la primera versión del Hit Score, que utilizaba normalización min-max directa, esta versión aplica una normalización robusta basada en percentiles. Cada variable se recorta entre los percentiles 1 y 99 antes de escalarse a una escala común entre 0 y 1.

Esta decisión es especialmente importante para `loudness`, donde los valores mínimos y máximos están alejados de la mayoría de la distribución. El clipping evita que unos pocos valores extremos dominen la escala de normalización.

In [6]:
correlation_rows = []

for feature in selected_features:
    corr_value = tracks_base.select(
        F.corr("popularity", feature).alias("corr")
    ).first()["corr"]
    
    correlation_rows.append((
        feature,
        float(corr_value),
        float(abs(corr_value))
    ))

total_abs_corr = sum(row[2] for row in correlation_rows)

weight_rows = [
    (
        feature,
        corr,
        abs_corr,
        abs_corr / total_abs_corr,
        "positiva" if corr >= 0 else "negativa"
    )
    for feature, corr, abs_corr in correlation_rows
]

feature_weights = spark.createDataFrame(
    weight_rows,
    [
        "feature",
        "correlation_with_popularity",
        "abs_correlation",
        "weight",
        "direction"
    ]
).orderBy(F.desc("weight"))

feature_weights.show(truncate=False)

+----------------+---------------------------+-------------------+-------------------+---------+
|feature         |correlation_with_popularity|abs_correlation    |weight             |direction|
+----------------+---------------------------+-------------------+-------------------+---------+
|acousticness    |-0.37096325577128686       |0.37096325577128686|0.25988213907708085|negativa |
|loudness        |0.32854329812262995        |0.32854329812262995|0.23016439975443223|positiva |
|energy          |0.30288206293384096        |0.30288206293384096|0.21218715648715247|positiva |
|instrumentalness|-0.23724359072707896       |0.23724359072707896|0.16620344705647533|negativa |
|danceability    |0.1877966149440401         |0.1877966149440401 |0.13156285762485917|positiva |
+----------------+---------------------------+-------------------+-------------------+---------+



### Pesos del Hit Score V2

En esta versión del `Hit Score`, los componentes no tienen el mismo peso. En lugar de usar un promedio simple, se asigna a cada variable un peso proporcional a su correlación absoluta con `popularity`.

Los resultados muestran que las variables con mayor peso son `acousticness`, `loudness` y `energy`. Esto indica que, dentro del dataset, estas variables presentan una relación lineal más fuerte con la popularidad.

La dirección de la correlación también se toma en cuenta. Las variables con correlación positiva, como `loudness`, `energy` y `danceability`, aportan más al score cuando sus valores son altos. En cambio, las variables con correlación negativa, como `acousticness` e `instrumentalness`, aportan más al score cuando sus valores son bajos.

Esta modificación permite que el `Hit Score V2` sea más consistente con los patrones observados durante el análisis exploratorio.

### crear componentes según la dirección de la correlación

In [7]:
weight_lookup = {
    row["feature"]: {
        "corr": row["correlation_with_popularity"],
        "weight": row["weight"]
    }
    for row in feature_weights.collect()
}

for feature in selected_features:
    corr = weight_lookup[feature]["corr"]
    
    if corr >= 0:
        component_expr = F.col(f"{feature}_norm_v2")
    else:
        component_expr = 1 - F.col(f"{feature}_norm_v2")
    
    tracks_scored_v2 = tracks_scored_v2.withColumn(
        f"{feature}_component_v2",
        component_expr
    )

In [9]:
tracks_scored_v2.select(
    "name",
    "popularity",
    "danceability_norm_v2",
    "danceability_component_v2",
    "energy_norm_v2",
    "energy_component_v2",
    "loudness_norm_v2",
    "loudness_component_v2",
    "acousticness_norm_v2",
    "acousticness_component_v2",
    "instrumentalness_norm_v2",
    "instrumentalness_component_v2"
).show(5, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------
 name                          | Pumping On Your Stereo 
 popularity                    | 50                     
 danceability_norm_v2          | 0.41095890410958913    
 danceability_component_v2     | 0.41095890410958913    
 energy_norm_v2                | 0.920626521325008      
 energy_component_v2           | 0.920626521325008      
 loudness_norm_v2              | 0.906137954584258      
 loudness_component_v2         | 0.906137954584258      
 acousticness_norm_v2          | 0.011008491644078154   
 acousticness_component_v2     | 0.9889915083559219     
 instrumentalness_norm_v2      | 0.09044585987261147    
 instrumentalness_component_v2 | 0.9095541401273886     
-RECORD 1-----------------------------------------------
 name                          | Mujhe Raat Din Bas     
 popularity                    | 50                     
 danceability_norm_v2          | 0.6205479452054794     
 danceability_component_v2     

### Construimos Score ponderado

In [10]:
from functools import reduce
from pyspark.sql import functions as F

weighted_terms = [
    F.col(f"{feature}_component_v2") * F.lit(float(weight_lookup[feature]["weight"]))
    for feature in selected_features
]

tracks_scored_v2 = (
    tracks_scored_v2
    .withColumn(
        "hit_score_weighted_raw",
        reduce(lambda a, b: a + b, weighted_terms)
    )
    .withColumn(
        "hit_score_weighted",
        F.round(F.col("hit_score_weighted_raw") * 100, 2)
    )
    .withColumn(
        "hit_score_weighted_class",
        F.when(F.col("hit_score_weighted") < 40, "bajo")
         .when(F.col("hit_score_weighted") < 70, "medio")
         .otherwise("alto")
    )
)

In [13]:
#canciones con su nuevo score
tracks_scored_v2.select(
    "name",
    "artists",
    "release_year",
    "popularity",
    "is_hit",
    "hit_score_weighted",
    "hit_score_weighted_class",
    "danceability_component_v2",
    "energy_component_v2",
    "loudness_component_v2",
    "acousticness_component_v2",
    "instrumentalness_component_v2"
).orderBy(F.desc("hit_score_weighted")).show(30, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------------------------------------
 name                          | HUMBLE. - SKRILLEX REMIX                                   
 artists                       | ['Skrillex', 'Kendrick Lamar']                             
 release_year                  | 2017                                                       
 popularity                    | 73                                                         
 is_hit                        | 1                                                          
 hit_score_weighted            | 98.78                                                      
 hit_score_weighted_class      | alto                                                       
 danceability_component_v2     | 1.0                                                        
 energy_component_v2           | 0.9523759127950049                                         
 loudness_component_v2         | 1.0                                  

In [14]:
tracks_scored_v2.select(
    F.round(
        F.corr("hit_score_weighted", "popularity"), 4
    ).alias("corr_hit_score_weighted_popularity")
).show()

+----------------------------------+
|corr_hit_score_weighted_popularity|
+----------------------------------+
|                            0.4154|
+----------------------------------+



### Validación inicial del Hit Score V2

Se calculó la correlación entre `hit_score_weighted` y `popularity` para evaluar si la nueva versión del score mantiene una relación positiva con la popularidad observada.

El resultado fue una correlación positiva de aproximadamente `0.4154`. Este valor es ligeramente superior al obtenido con la primera versión del `Hit Score`, que era aproximadamente `0.4115`.

Aunque la mejora en correlación no es grande, la versión V2 es metodológicamente más robusta porque incorpora normalización con clipping por percentiles y ponderación de variables según su correlación absoluta con `popularity`.

Esto sugiere que el `Hit Score V2` conserva la capacidad de ordenar canciones en una dirección coherente con la popularidad, pero con una construcción más justificada.

In [15]:
hit_score_weighted_bins = (
    tracks_scored_v2
    .withColumn(
        "hit_score_weighted_range",
        F.when(F.col("hit_score_weighted") < 40, "00-39")
         .when(F.col("hit_score_weighted") < 50, "40-49")
         .when(F.col("hit_score_weighted") < 60, "50-59")
         .when(F.col("hit_score_weighted") < 70, "60-69")
         .when(F.col("hit_score_weighted") < 80, "70-79")
         .when(F.col("hit_score_weighted") < 90, "80-89")
         .otherwise("90-100")
    )
    .groupBy("hit_score_weighted_range")
    .agg(
        F.count("*").alias("num_canciones"),
        F.round(F.avg("hit_score_weighted"), 2).alias("avg_hit_score_weighted"),
        F.round(F.avg("popularity"), 2).alias("avg_popularity"),
        F.round(F.avg("is_hit") * 100, 2).alias("hit_rate_pct")
    )
    .orderBy("hit_score_weighted_range")
)

hit_score_weighted_bins.show(truncate=False)

+------------------------+-------------+----------------------+--------------+------------+
|hit_score_weighted_range|num_canciones|avg_hit_score_weighted|avg_popularity|hit_rate_pct|
+------------------------+-------------+----------------------+--------------+------------+
|00-39                   |81981        |27.02                 |14.56         |0.14        |
|40-49                   |68031        |45.29                 |18.71         |0.34        |
|50-59                   |83386        |55.1                  |24.17         |0.49        |
|60-69                   |94996        |65.13                 |28.59         |0.89        |
|70-79                   |115219       |75.1                  |32.26         |1.62        |
|80-89                   |119138       |84.82                 |36.33         |2.64        |
|90-100                  |23513        |91.86                 |39.18         |3.0         |
+------------------------+-------------+----------------------+--------------+--

### Validación por rangos del Hit Score V2

Para validar el comportamiento del `Hit Score V2`, se agruparon las canciones en rangos de puntuación y se calculó la popularidad promedio y la proporción de canciones clasificadas como hit.

Los resultados muestran una tendencia positiva: conforme aumenta el rango de `hit_score_weighted`, también aumenta la popularidad promedio. Las canciones en el rango 00-39 tienen una popularidad promedio de 14.56, mientras que las canciones en el rango 90-100 alcanzan una popularidad promedio de 39.18.

También se observa que la proporción de hits aumenta en los rangos superiores del score. Esto sugiere que el `Hit Score V2` captura parcialmente un perfil acústico asociado con canciones populares.

Aunque la proporción de hits sigue siendo baja, esto es esperado porque la definición de hit usada en el proyecto es estricta: `popularity >= 70`, umbral cercano al percentil 99 de la distribución de popularidad.

En comparación con la primera versión, el `Hit Score V2` mantiene una relación positiva con la popularidad y además es metodológicamente más robusto, ya que usa clipping por percentiles y pesos basados en correlación.

Como vimos que las recomendaciones podían salir muy antiguas, vamos a crear una puntuación de recencia. Esta no reemplaza al score acústico; solo permite construir una versión ajustada temporalmente.

In [16]:
import math
from pyspark.sql import functions as F

max_year = tracks_scored_v2.select(
    F.max("release_year").alias("max_year")
).first()["max_year"]

half_life = 10
lambda_decay = math.log(2) / half_life

tracks_scored_v2 = (
    tracks_scored_v2
    .withColumn(
        "song_age",
        F.when(
            F.col("release_year").isNotNull(),
            F.lit(max_year) - F.col("release_year")
        ).otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "recency_weight",
        F.when(
            F.col("song_age").isNotNull(),
            F.exp(-F.lit(lambda_decay) * F.col("song_age"))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "recency_score",
        F.round(F.col("recency_weight") * 100, 2)
    )
    .withColumn(
        "hit_score_recency_adjusted",
        F.round(
            0.80 * F.col("hit_score_weighted") +
            0.20 * F.col("recency_score"),
            2
        )
    )
)

In [18]:
#resultados
tracks_scored_v2.select(
    "name",
    "artists",
    "release_year",
    "song_age",
    "popularity",
    "hit_score_weighted",
    "recency_score",
    "hit_score_recency_adjusted"
).orderBy(F.desc("hit_score_recency_adjusted")).show(30, truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------------------------
 name                       | Pepa & Palo - Remix                                                                
 artists                    | ['Kiko el Crazy', 'Haraca Kiko', 'El Fother', 'Bulova', 'Bulin 47', 'Shadow Blow'] 
 release_year               | 2021                                                                               
 song_age                   | 0                                                                                  
 popularity                 | 47                                                                                 
 hit_score_weighted         | 96.9                                                                               
 recency_score              | 100.0                                                                              
 hit_score_recency_adjusted | 97.52                                                     

## Ajuste temporal del Hit Score

Durante la revisión cualitativa de las recomendaciones se observó que algunas canciones sugeridas eran muy antiguas. Esto ocurre porque el `Hit Score V2` mide similitud acústica con el perfil de canciones populares, pero no considera el año de lanzamiento.

Para atender esta limitación, se construye una versión ajustada por recencia llamada `hit_score_recency_adjusted`.

Esta métrica combina el `hit_score_weighted`, que representa el perfil acústico, con un `recency_score`, que favorece canciones más recientes mediante una función de decaimiento exponencial.

El ajuste temporal no reemplaza al score acústico. Más bien, ofrece una segunda forma de ordenar canciones cuando se desea priorizar recomendaciones más recientes.

La fórmula usada es:

`hit_score_recency_adjusted = 0.80 * hit_score_weighted + 0.20 * recency_score`

De esta forma, la mayor parte del score sigue dependiendo de las características acústicas, pero se penalizan suavemente canciones demasiado antiguas.

## Hidden Gems

Primero trabajaremos con Hidden Gems acústicas

In [20]:
hidden_gems_v2_acoustic = (
    tracks_scored_v2
    .filter(
        (F.col("popularity") < 40) &
        (F.col("hit_score_weighted") >= 80)
    )
    .select(
        "id",
        "name",
        "artists",
        "release_year",
        "popularity",
        "hit_score_weighted",
        "recency_score",
        "hit_score_recency_adjusted",
        "danceability",
        "energy",
        "loudness",
        "acousticness",
        "instrumentalness"
    )
    .orderBy(F.desc("hit_score_weighted"))
)

hidden_gems_v2_acoustic.show(20, truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------
 id                         | 3O6x1B9uvMAnZJpLfzWzCM                         
 name                       | Critical Beatdown                              
 artists                    | ["Ultramagnetic MC's"]                         
 release_year               | 1988                                           
 popularity                 | 36                                             
 hit_score_weighted         | 98.39                                          
 recency_score              | 10.15                                          
 hit_score_recency_adjusted | 80.74                                          
 danceability               | 0.933                                          
 energy                     | 0.959                                          
 loudness                   | -3.682                                         
 acousticness               | 0.00337                           

In [22]:
hidden_gems_v2_modern = (
    tracks_scored_v2
    .filter(
        (F.col("popularity") < 40) &
        (F.col("release_year") >= 2010) &
        (F.col("hit_score_recency_adjusted") >= 70)
    )
    .select(
        "id",
        "name",
        "artists",
        "release_year",
        "popularity",
        "hit_score_weighted",
        "recency_score",
        "hit_score_recency_adjusted",
        "danceability",
        "energy",
        "loudness",
        "acousticness",
        "instrumentalness"
    )
    .orderBy(F.desc("hit_score_recency_adjusted"))
)

hidden_gems_v2_modern.show(30, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------------
 id                         | 1EhQAE3gcsY8tptV9guCxT                                                  
 name                       | Bonita - Remix                                                          
 artists                    | ['J Balvin', 'Jowell & Randy', 'Nicky Jam', 'Wisin', 'Yandel', 'Ozuna'] 
 release_year               | 2021                                                                    
 popularity                 | 0                                                                       
 hit_score_weighted         | 96.46                                                                   
 recency_score              | 100.0                                                                   
 hit_score_recency_adjusted | 97.17                                                                   
 danceability               | 0.796                                      